# Data Ingestion, Cleaning & Preprocessing with Pandas

## Retail Store Sales Dataset

### Objective

The objective of this project is to ingest, inspect, clean, standardize,
and preprocess a real-world retail sales dataset using Python and Pandas.

The analysis focuses on:

- Identifying missing values
- Detecting duplicate records
- Handling inconsistent data types
- Identifying invalid values and outliers
- Performing feature engineering
- Extracting year and month from dates
- Creating useful business metrics
- Comparing the dataset before and after cleaning
- Exporting the cleaned dataset as a CSV file

### Tools Used

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Google Colab

In [5]:
from google.colab import files

uploaded = files.upload()

Saving retail_store_sales.csv to retail_store_sales (1).csv


In [7]:
import pandas as pd
import numpy as np

file_name = "retail_store_sales.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("File name:", file_name)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully!
File name: retail_store_sales.csv
Number of rows: 12575
Number of columns: 11


In [8]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [10]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values in each column:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

Number of duplicate rows: 0


In [11]:
print("Shape:", df.shape)
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(i, "->", col)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (12575, 11)

Column names:
1 -> Transaction ID
2 -> Customer ID
3 -> Category
4 -> Item
5 -> Price Per Unit
6 -> Quantity
7 -> Total Spent
8 -> Payment Method
9 -> Location
10 -> Transaction Date
11 -> Discount Applied

Data types:
Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object

Missing values:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

Duplicate rows: 0


In [13]:
# Make a copy of the original dataset
clean_data = df.copy()

# Standardize column names
clean_data.columns = (
    clean_data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Standardized column names:")
print(clean_data.columns.tolist())

Standardized column names:
['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


In [14]:
# Convert numeric columns to numeric type
numeric_columns = [
    "price_per_unit",
    "quantity",
    "total_spent"
]

for col in numeric_columns:
    clean_data[col] = pd.to_numeric(
        clean_data[col],
        errors="coerce"
    )

# Convert transaction date to datetime
clean_data["transaction_date"] = pd.to_datetime(
    clean_data["transaction_date"],
    errors="coerce"
)

print("Data types after conversion:")
print(clean_data.dtypes)

Data types after conversion:
transaction_id              object
customer_id                 object
category                    object
item                        object
price_per_unit             float64
quantity                   float64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
discount_applied            object
dtype: object


In [16]:
# Handle missing Discount Applied values
clean_data["discount_applied"] = (
    clean_data["discount_applied"]
    .fillna(False)
    .astype(bool)
)

print("Discount Applied after cleaning:")
print(clean_data["discount_applied"].value_counts(dropna=False))

Discount Applied after cleaning:
discount_applied
False    8356
True     4219
Name: count, dtype: int64


/tmp/ipykernel_2111/3698559626.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [18]:
# Handle missing Item values
clean_data["item"] = clean_data["item"].fillna("Unknown")

print("Missing Item values:",
      clean_data["item"].isna().sum())

Missing Item values: 0


In [20]:
# Handle missing numeric values using median imputation

numeric_columns = [
    "price_per_unit",
    "quantity",
    "total_spent"
]

for col in numeric_columns:
    median_value = clean_data[col].median()
    clean_data[col] = clean_data[col].fillna(median_value)

print("Missing numeric values after imputation:")
print(clean_data[numeric_columns].isna().sum())

Missing numeric values after imputation:
price_per_unit    0
quantity          0
total_spent       0
dtype: int64


In [21]:
print("Missing values after cleaning:")
print(clean_data.isna().sum())

print("\nDuplicate rows:", clean_data.duplicated().sum())

print("\nDataset shape:")
print(clean_data.shape)

Missing values after cleaning:
transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
dtype: int64

Duplicate rows: 0

Dataset shape:
(12575, 11)


In [23]:
# Check for invalid numeric values

print("Negative values:")
print("Price Per Unit:", (clean_data["price_per_unit"] < 0).sum())
print("Quantity:", (clean_data["quantity"] < 0).sum())
print("Total Spent:", (clean_data["total_spent"] < 0).sum())

print("\nZero values:")
print("Price Per Unit:", (clean_data["price_per_unit"] == 0).sum())
print("Quantity:", (clean_data["quantity"] == 0).sum())
print("Total Spent:", (clean_data["total_spent"] == 0).sum())

print("\nSummary statistics:")
print(clean_data[[
    "price_per_unit",
    "quantity",
    "total_spent"
]].describe())

Negative values:
Price Per Unit: 0
Quantity: 0
Total Spent: 0

Zero values:
Price Per Unit: 0
Quantity: 0
Total Spent: 0

Summary statistics:
       price_per_unit      quantity   total_spent
count    12575.000000  12575.000000  12575.000000
mean        23.348191      5.558648    128.636581
std         10.480413      2.790160     92.557580
min          5.000000      1.000000      5.000000
25%         14.000000      3.000000     55.000000
50%         23.000000      6.000000    108.500000
75%         32.000000      8.000000    184.000000
max         41.000000     10.000000    410.000000


In [25]:
# Detect outliers using the IQR method

numeric_columns = [
    "price_per_unit",
    "quantity",
    "total_spent"
]

print("Outlier analysis using IQR")
print("=" * 40)

for col in numeric_columns:
    Q1 = clean_data[col].quantile(0.25)
    Q3 = clean_data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (clean_data[col] < lower_bound) |
        (clean_data[col] > upper_bound)
    ).sum()

    print(f"\n{col}")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Number of outliers: {outliers}")

Outlier analysis using IQR

price_per_unit
Q1: 14.00
Q3: 32.00
IQR: 18.00
Lower bound: -13.00
Upper bound: 59.00
Number of outliers: 0

quantity
Q1: 3.00
Q3: 8.00
IQR: 5.00
Lower bound: -4.50
Upper bound: 15.50
Number of outliers: 0

total_spent
Q1: 55.00
Q3: 184.00
IQR: 129.00
Lower bound: -138.50
Upper bound: 377.50
Number of outliers: 157


In [27]:
# Inspect Total Spent outliers

Q1 = clean_data["total_spent"].quantile(0.25)
Q3 = clean_data["total_spent"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

outlier_data = clean_data[
    clean_data["total_spent"] > upper_bound
].copy()

print("Number of Total Spent outliers:", len(outlier_data))
print("\nHighest Total Spent values:")
print(
    outlier_data[
        ["transaction_id", "customer_id", "item",
         "price_per_unit", "quantity", "total_spent"]
    ]
    .sort_values("total_spent", ascending=False)
    .head(10)
)

Number of Total Spent outliers: 157

Highest Total Spent values:
     transaction_id customer_id          item  price_per_unit  quantity  \
27      TXN_1599706     CUST_14   Item_25_FUR            41.0      10.0   
133     TXN_2953434     CUST_25   Item_25_FUR            41.0      10.0   
1060    TXN_3710081     CUST_25   Item_25_FUR            41.0      10.0   
869     TXN_1814138     CUST_06   Item_25_BEV            41.0      10.0   
339     TXN_4374445     CUST_12  Item_25_FOOD            41.0      10.0   
1468    TXN_1938135     CUST_17   Item_25_BUT            41.0      10.0   
1983    TXN_6924479     CUST_21   Item_25_PAT            41.0      10.0   
1950    TXN_1860120     CUST_01   Item_25_FUR            41.0      10.0   
2153    TXN_5338814     CUST_06       Unknown            23.0      10.0   
1568    TXN_8048041     CUST_14   Item_25_BUT            41.0      10.0   

      total_spent  
27          410.0  
133         410.0  
1060        410.0  
869         410.0  
339      

In [29]:
# Feature engineering from Transaction Date

clean_data["year"] = clean_data["transaction_date"].dt.year
clean_data["month"] = clean_data["transaction_date"].dt.month
clean_data["month_name"] = clean_data["transaction_date"].dt.month_name()
clean_data["day_of_week"] = clean_data["transaction_date"].dt.day_name()

print("Date features created successfully!")

print(
    clean_data[
        ["transaction_date", "year", "month",
         "month_name", "day_of_week"]
    ].head()
)

Date features created successfully!
  transaction_date  year  month month_name day_of_week
0       2024-04-08  2024      4      April      Monday
1       2023-07-23  2023      7       July      Sunday
2       2022-10-05  2022     10    October   Wednesday
3       2022-05-07  2022      5        May    Saturday
4       2022-10-02  2022     10    October      Sunday


In [31]:
# Before vs After Cleaning Comparison

print("BEFORE CLEANING")
print("================")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nAFTER CLEANING")
print("================")
print("Rows:", clean_data.shape[0])
print("Columns:", clean_data.shape[1])
print("Missing values:", clean_data.isna().sum().sum())
print("Duplicate rows:", clean_data.duplicated().sum())

print("\nFINAL DATA TYPES")
print("================")
print(clean_data.dtypes)

BEFORE CLEANING
Rows: 12575
Columns: 11
Missing values: 7229
Duplicate rows: 0

AFTER CLEANING
Rows: 12575
Columns: 15
Missing values: 0
Duplicate rows: 0

FINAL DATA TYPES
transaction_id              object
customer_id                 object
category                    object
item                        object
price_per_unit             float64
quantity                   float64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
discount_applied              bool
year                         int32
month                        int32
month_name                  object
day_of_week                 object
dtype: object


In [33]:
# Export cleaned dataset

output_file = "clean_retail_sales.csv"

clean_data.to_csv(output_file, index=False)

print("Clean dataset exported successfully!")
print("File name:", output_file)
print("Rows:", clean_data.shape[0])
print("Columns:", clean_data.shape[1])

Clean dataset exported successfully!
File name: clean_retail_sales.csv
Rows: 12575
Columns: 15


In [34]:
from google.colab import files

files.download("clean_retail_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>